# Time series benchmarking tutorial (Hydrology CAMELS US with Chronos 2)

This is the **main DS5110 final notebook**.

## Stage 1 (this update): Data input and preprocessing

- Input file: `data/raw/BasicInputTimeSeries_us.csv`
- Task: single-step forecasting setup preparation
- Input mode: strictly univariate (`QObs(mm/d)` only)
- Lookback window: 30
- Split: 80% train / 10% val / 10% test (chronological per basin)

In this stage, we focus only on loading, validating, cleaning, splitting, and saving processed artifacts for later Chronos-2 evaluation.


In [ ]:
# Install dependencies (Colab-friendly)
import subprocess
import sys
from pathlib import Path

req_candidates = [
    Path('requirements.txt'),
    Path('/content/requirements.txt'),
    Path('/content/timeseries_benchmark_tutorial/requirements.txt'),
    Path('/content/drive/MyDrive/timeseries_benchmark_tutorial/requirements.txt'),
    Path('../requirements.txt'),
]

req_path = next((p for p in req_candidates if p.exists()), None)
if req_path is not None:
    print(f'Installing from: {req_path.resolve()}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_path)])
else:
    print('requirements.txt not found. Installing fallback packages...')
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'pandas>=2.0',
        'numpy>=1.24',
        'matplotlib>=3.7',
        'pyarrow>=14.0',
        'chronos-forecasting>=2.0',
        'torch>=2.1',
        'tqdm>=4.66',
        'pyyaml>=6.0',
    ])

print('Dependency installation complete.')

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Optional Colab Drive mount:
# from google.colab import drive
# drive.mount('/content/drive')

# ----------------------------
# User-editable runtime config
# ----------------------------
INPUT_CSV_PATH = 'data/raw/BasicInputTimeSeries_us.csv'  # In Colab, update if needed (e.g., '/content/BasicInputTimeSeries_us.csv')
LOOKBACK_WINDOW = 30
FORECAST_HORIZON = 1
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

TIMESTAMP_COL = 'Year_Mnth_Day'
ID_COL = 'basin_id'
TARGET_COL = 'QObs(mm/d)'
REQUIRED_COLUMNS = [TIMESTAMP_COL, ID_COL, TARGET_COL]

OUTPUT_ROOT = Path('.')
OUTPUT_DIRS = [
    OUTPUT_ROOT / 'metadata',
    OUTPUT_ROOT / 'results',
    OUTPUT_ROOT / 'results' / 'figures',
    OUTPUT_ROOT / 'data' / 'processed',
]
for d in OUTPUT_DIRS:
    d.mkdir(parents=True, exist_ok=True)

print('Configured input path:', INPUT_CSV_PATH)
print('Output root:', OUTPUT_ROOT.resolve())

In [ ]:
import sys

# Make src/ imports work in both local and Colab contexts.
src_candidates = [Path('src'), Path('/content/timeseries_benchmark_tutorial/src'), Path('../src')]
for src in src_candidates:
    if src.exists():
        sys.path.insert(0, str(src.resolve()))
        print('Using src path:', src.resolve())
        break

from data_utils import (
    DataSpec,
    chronological_split_by_id,
    load_and_prepare_csv,
    to_chronos_df,
)

In [ ]:
# Load quick preview and validate required columns
if not INPUT_CSV_PATH.exists():
    raise FileNotFoundError(
        'Input CSV not found. Set env var INPUT_CSV_PATH or place file at data/raw/BasicInputTimeSeries_us.csv'
    )

preview_df = pd.read_csv(INPUT_CSV_PATH, nrows=5)
print('Preview columns:', list(preview_df.columns))
print(preview_df.head())

missing_required = [c for c in REQUIRED_COLUMNS if c not in preview_df.columns]
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

print('Required columns found:', REQUIRED_COLUMNS)

In [ ]:
# Load full dataset and preprocess
spec = DataSpec(
    timestamp_col=TIMESTAMP_COL,
    id_col=ID_COL,
    target_col=TARGET_COL,
    drop_unnamed_first_col=True,  # user requirement
)

raw_df = load_and_prepare_csv(INPUT_CSV_PATH, spec)

print('Prepared rows:', len(raw_df))
print('Prepared basins:', raw_df[ID_COL].nunique())
print('Date range:', raw_df[TIMESTAMP_COL].min(), 'to', raw_df[TIMESTAMP_COL].max())
print(raw_df[[TIMESTAMP_COL, ID_COL, TARGET_COL]].head())

In [ ]:
# Split each basin chronologically: 80/10/10
splits = chronological_split_by_id(
    raw_df,
    id_col=ID_COL,
    timestamp_col=TIMESTAMP_COL,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
)

train_df_raw = splits['train']
val_df_raw = splits['val']
test_df_raw = splits['test']

print('Train rows:', len(train_df_raw))
print('Val rows:', len(val_df_raw))
print('Test rows:', len(test_df_raw))

# Convert to Chronos long format (still only data prep stage)
train_df = to_chronos_df(train_df_raw, id_col=ID_COL, timestamp_col=TIMESTAMP_COL, target_col=TARGET_COL)
val_df = to_chronos_df(val_df_raw, id_col=ID_COL, timestamp_col=TIMESTAMP_COL, target_col=TARGET_COL)
test_df = to_chronos_df(test_df_raw, id_col=ID_COL, timestamp_col=TIMESTAMP_COL, target_col=TARGET_COL)

print('\nChronos-formatted sample:')
print(train_df.head())

In [ ]:
# Save processed artifacts for reproducible next stages
processed_dir = OUTPUT_ROOT / 'data' / 'processed'

train_df.to_parquet(processed_dir / 'train_chronos.parquet', index=False)
val_df.to_parquet(processed_dir / 'val_chronos.parquet', index=False)
test_df.to_parquet(processed_dir / 'test_chronos.parquet', index=False)

# Small CSV heads for quick inspection
train_df.head(1000).to_csv(processed_dir / 'train_preview_1000.csv', index=False)
val_df.head(1000).to_csv(processed_dir / 'val_preview_1000.csv', index=False)
test_df.head(1000).to_csv(processed_dir / 'test_preview_1000.csv', index=False)

print('Saved processed splits to:', processed_dir.resolve())

In [ ]:
# Quick data sanity visualization (preprocessing stage)
# Plot target over time for one example basin.
example_id = str(train_df['id'].iloc[0])
example_series = train_df[train_df['id'] == example_id].sort_values('timestamp').tail(365)

plt.figure(figsize=(12, 4))
plt.plot(example_series['timestamp'], example_series['target'])
plt.title(f'Example basin {example_id}: last 365 train points of {TARGET_COL}')
plt.xlabel('Date')
plt.ylabel(TARGET_COL)
plt.tight_layout()

prep_fig_path = OUTPUT_ROOT / 'results' / 'figures' / 'preprocess_example_series.png'
plt.savefig(prep_fig_path, dpi=150)
plt.show()
print('Saved:', prep_fig_path)

In [ ]:
# Save preprocessing summary for reproducibility
preprocess_summary = {
    'project_root': str(PROJECT_ROOT),
    'input_csv_path': str(INPUT_CSV_PATH),
    'timestamp_column': TIMESTAMP_COL,
    'id_column': ID_COL,
    'target_column': TARGET_COL,
    'drop_unnamed_first_column': True,
    'fill_strategy': 'forward_fill_within_basin',
    'train_ratio': TRAIN_RATIO,
    'val_ratio': VAL_RATIO,
    'test_ratio': TEST_RATIO,
    'lookback_window': LOOKBACK_WINDOW,
    'forecast_horizon': FORECAST_HORIZON,
    'prepared_rows': int(len(raw_df)),
    'prepared_basins': int(raw_df[ID_COL].nunique()),
    'train_rows': int(len(train_df)),
    'val_rows': int(len(val_df)),
    'test_rows': int(len(test_df)),
}

summary_path = OUTPUT_ROOT / 'results' / 'preprocessing_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(preprocess_summary, f, indent=2)

print('Saved:', summary_path)
print(json.dumps(preprocess_summary, indent=2))

In [ ]:
# (Next stage placeholder)
# Chronos-2 model loading and rolling one-step inference will be added next.
print('Stage 1 complete: input + preprocessing artifacts are ready.')
print('Next stage: model inference, test RMSE, and Actual vs Predicted.')

In [ ]:
# Update metadata files with observed dataset counts
# This keeps machine-readable metadata consistent with the real preprocessed input.
dataset_card_path = OUTPUT_ROOT / 'metadata' / 'dataset_card.json'
ts_chars_path = OUTPUT_ROOT / 'metadata' / 'ts_characteristics.json'
experiment_cfg_path = OUTPUT_ROOT / 'metadata' / 'experiment_config.json'

if dataset_card_path.exists():
    with open(dataset_card_path, 'r', encoding='utf-8') as f:
        dataset_card = json.load(f)
    dataset_card['number_of_locations'] = int(raw_df[ID_COL].nunique())
    dataset_card['number_of_time_points'] = int(len(raw_df))
    with open(dataset_card_path, 'w', encoding='utf-8') as f:
        json.dump(dataset_card, f, indent=2)

if ts_chars_path.exists():
    with open(ts_chars_path, 'r', encoding='utf-8') as f:
        ts_chars = json.load(f)
    ts_chars['number_of_instances'] = int(raw_df[ID_COL].nunique())
    with open(ts_chars_path, 'w', encoding='utf-8') as f:
        json.dump(ts_chars, f, indent=2)

if experiment_cfg_path.exists():
    with open(experiment_cfg_path, 'r', encoding='utf-8') as f:
        exp_cfg = json.load(f)
    exp_cfg['lookback_window'] = int(LOOKBACK_WINDOW)
    exp_cfg['forecast_horizon'] = int(FORECAST_HORIZON)
    exp_cfg['evaluation_metrics'] = ['rmse']
    with open(experiment_cfg_path, 'w', encoding='utf-8') as f:
        json.dump(exp_cfg, f, indent=2)

print('Metadata refresh complete.')

## Stage 1 outputs (input + preprocessing)

- `data/processed/train_chronos.parquet`
- `data/processed/val_chronos.parquet`
- `data/processed/test_chronos.parquet`
- `data/processed/*_preview_1000.csv`
- `results/preprocessing_summary.json`
- `results/figures/preprocess_example_series.png`
- refreshed metadata files in `metadata/`

Next stage will add:
- Chronos-2 inference
- overall test RMSE
- `results/benchmark_results.json`
- `results/figures/actual_vs_predicted.png`
